## BLOQUE 0 — Setup y configuración del entorno 
, 
,Carga la configuración, agrega SSL bypass para la red corporativa 
,y actualiza `config.py` con el parámetro `VERIFY_SSL`. 



In [69]:
import sys, importlib 
from pathlib import Path 
 
PROJECT_ROOT = Path.cwd() 
for ruta in (str(PROJECT_ROOT), str(PROJECT_ROOT / "src")): 
    if ruta not in sys.path: 
        sys.path.insert(0, ruta) 
 
import config 
importlib.reload(config) 
config.ensure_dirs() 
 
from utils.logger import get_logger 
log = get_logger("01_scraping") 
log.info("Notebook 01_scraping iniciado") 
 
print(f"PROJECT_ROOT : {PROJECT_ROOT}") 
print(f"VERIFY_SSL   : {config.VERIFY_SSL}") 
print(f"Fuentes activas: {list(config.fuentes_activas().keys())}") 


10:02:29 | INFO     | 01_scraping | Notebook 01_scraping iniciado


PROJECT_ROOT : c:\Users\User\Documents\normas-grc-pipeline-main
VERIFY_SSL   : False
Fuentes activas: ['SFC', 'BANREP', 'MINHAC', 'DIAN', 'URF']


In [70]:
import subprocess, sys
# PC personal: pip normal (sin Artifactory)
paquetes = ['selenium', 'requests', 'beautifulsoup4', 'lxml', 'pandas', 'openpyxl', 'loguru', 'python-dotenv']
for pkg in paquetes:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], capture_output=True)
    estado = '✓' if r.returncode == 0 else '✗'
    print(f'  {estado} {pkg}')
print('\nDependencias listas.')


  ✓ selenium
  ✓ requests
  ✓ beautifulsoup4
  ✓ lxml
  ✓ pandas
  ✓ openpyxl
  ✓ loguru
  ✓ python-dotenv

Dependencias listas.


In [71]:
import subprocess, sys
# pdfminer.six — pip normal
r = subprocess.run([sys.executable, '-m', 'pip', 'install', 'pdfminer.six', '-q'], capture_output=True)
print('✓ pdfminer.six' if r.returncode == 0 else '✗ pdfminer.six — ' + r.stderr.decode()[:100])


✓ pdfminer.six


In [72]:
# ── Asegurar que VERIFY_SSL=false esté en el .env ────────────────── 
# La red corporativa de Bancolombia intercepta SSL — necesitamos bypassear. 
env_path = PROJECT_ROOT / ".env" 
env_text = env_path.read_text(encoding="utf-8") if env_path.exists() else "" 
 
if "VERIFY_SSL" not in env_text: 
    env_path.write_text(env_text.rstrip() + "\nVERIFY_SSL=false\n", encoding="utf-8") 
    print("\u2713 VERIFY_SSL=false agregado al .env") 
else: 
    # Asegurarse de que no sea true 
    import re 
    env_text = re.sub(r"VERIFY_SSL=.*", "VERIFY_SSL=false", env_text) 
    env_path.write_text(env_text, encoding="utf-8") 
    print("\u2713 VERIFY_SSL=false confirmado en .env") 
 
# Recargar config para que tome el nuevo valor 
importlib.reload(config) 
print(f"config.VERIFY_SSL = {config.VERIFY_SSL}") 
 
# Suprimir warnings de SSL a nivel de proceso 
import urllib3 
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning) 
print("\u2713 Warnings de SSL suprimidos") 


✓ VERIFY_SSL=false confirmado en .env
config.VERIFY_SSL = False
✓ Warnings de SSL suprimidos


In [73]:
import requests, urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'

fuentes_test = {
    'SFC':    'https://www.superfinanciera.gov.co',
    'BanRep': 'https://www.banrep.gov.co',
    'MinHac': 'https://www.minhacienda.gov.co',
    'DIAN':   'https://www.dian.gov.co',
}

print('Test de conectividad:')
print('─' * 55)
for nombre, url in fuentes_test.items():
    try:
        r = requests.get(url, timeout=10, verify=config.VERIFY_SSL,
                         headers={'User-Agent': USER_AGENT})
        print(f'  {nombre:10s} ✓ HTTP {r.status_code} — accesible')
    except requests.exceptions.SSLError as e:
        print(f'  {nombre:10s} ⚠  SSL error — {str(e)[:60]}')
    except requests.exceptions.ConnectionError as e:
        print(f'  {nombre:10s} ✗  Sin conexión — {str(e)[:80]}')
    except Exception as e:
        print(f'  {nombre:10s} ✗  Error: {str(e)[:60]}')


Test de conectividad:
───────────────────────────────────────────────────────
  SFC        ✓ HTTP 200 — accesible
  BanRep     ✓ HTTP 200 — accesible
  MinHac     ✓ HTTP 200 — accesible
  DIAN       ✓ HTTP 200 — accesible


## BLOQUE 1 — `base_scraper.py` (Clase abstracta) 
 
La clase base que todos los scrapers heredan. Contiene: 
- Sesión HTTP con SSL configurable (bypass para proxy corporativo) 
- Retry automático con backoff exponencial (tenacity) 
- Delay de cortesía entre requests 
- Normalización de fechas al formato DD/MM/YYYY 
- Constructor de registros con schema raw 
- Método `run()` orquestador 
 
**No se modifica** cuando agregas nuevas fuentes. 


In [ ]:
ruta = PROJECT_ROOT / 'src/scraper/base_scraper.py'
print(f"✓ src/scraper/base_scraper.py" if ruta.exists() else "✗ FALTA src/scraper/base_scraper.py")


✓ src/scraper/base_scraper.py


In [75]:
# ── Test: verificar que base_scraper importa correctamente ────────── 
import importlib.util, sys 
 
spec = importlib.util.spec_from_file_location( 
    "base_scraper", 
    PROJECT_ROOT / "src" / "scraper" / "base_scraper.py" 
) 
mod = importlib.util.module_from_spec(spec) 
spec.loader.exec_module(mod) 
 
# Verificar que BaseScraper tiene los métodos abstractos 
BaseScraper = mod.BaseScraper 
assert hasattr(BaseScraper, "fetch_index"), "Falta fetch_index" 
assert hasattr(BaseScraper, "parse"),       "Falta parse" 
assert hasattr(BaseScraper, "run"),         "Falta run" 
assert hasattr(BaseScraper, "_get"),        "Falta _get" 
assert hasattr(BaseScraper, "_build_record"), "Falta _build_record" 
 
print("\u2713 base_scraper.py cargado correctamente") 
print(f"  Métodos abstractos: fetch_index, parse") 
print(f"  Métodos heredados  : _get, _soup, _hash, _norm_date, _build_record, run") 
print(f"  SSL bypass         : VERIFY_SSL = {config.VERIFY_SSL}") 


✓ base_scraper.py cargado correctamente
  Métodos abstractos: fetch_index, parse
  Métodos heredados  : _get, _soup, _hash, _norm_date, _build_record, run
  SSL bypass         : VERIFY_SSL = False


## BLOQUE 2 — `sfc_scraper.py` (Superintendencia Financiera de Colombia) 
 
**Fuente:** https://www.superfinanciera.gov.co 
 
**Qué extrae:** 
- Circulares Externas (2023-2024) 
- Resoluciones (2023-2024) 
- Cartas Circulares (2024) 
 
**Para agregar más años:** agregar la URL del año al listado `INDEX_URLS` dentro del archivo. 
 
**Patrón de URL:** `/publicaciones/{ID}/{slug}/` 


In [ ]:
#import base64 
#_b64 = 'IiIiCnNyYy9zY3JhcGVyL3NmY19zY3JhcGVyLnB5IOKAlCBTdXBlcmludGVuZGVuY2lhIEZpbmFuY2llcmEgZGUgQ29sb21iaWEuCgpGdWVudGU6IGh0dHBzOi8vd3d3LnN1cGVyZmluYW5jaWVyYS5nb3YuY28KRXh0cmFlOiBDaXJjdWxhcmVzIEV4dGVybmFzLCBSZXNvbHVjaW9uZXMsIENhcnRhcyBDaXJjdWxhcmVzCiIiIgppbXBvcnQgcmUKZnJvbSB0eXBpbmcgaW1wb3J0IE9wdGlvbmFsCmZyb20gYmFzZV9zY3JhcGVyIGltcG9ydCBCYXNlU2NyYXBlcgppbXBvcnQgY29uZmlnCgoKY2xhc3MgU0ZDU2NyYXBlcihCYXNlU2NyYXBlcik6CiAgICAiIiIKICAgIFNjcmFwZXIgcGFyYSBsYSBTdXBlcmludGVuZGVuY2lhIEZpbmFuY2llcmEgZGUgQ29sb21iaWEgKFNGQykuCgogICAgRXN0cnVjdHVyYSBkZWwgc2l0aW86CiAgICAgICAgUMOhZ2luYXMgZGUgbGlzdGFkbzogL3B1YmxpY2FjaW9uZXMve0lEfS97c2x1Z30vCiAgICAgICAgTm9ybWFzIGluZGl2aWR1YWxlczogbWlzbW8gcGF0csOzbiBVUkwKICAgICAgICBBw7FvcyBkaXNwb25pYmxlczogYWdyZWdhciBtw6FzIGVudHJhZGFzIGVuIElOREVYX1VSTFMKICAgICIiIgoKICAgIFNPVVJDRV9DT0RFID0gIlNGQyIKCiAgICAjIOKUgOKUgCBVUkxzIGRlIGxpc3RhZG8gcG9yIHRpcG8geSBhw7FvIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgIyBQYXJhIGFncmVnYXIgbcOhcyBhw7FvcyBvIHRpcG9zLCBhw7FhZGlyIGxhIFVSTCBjb3JyZXNwb25kaWVudGUuCiAgICBJTkRFWF9VUkxTID0gWwogICAgICAgICJodHRwczovL3d3dy5zdXBlcmZpbmFuY2llcmEuZ292LmNvL3B1YmxpY2FjaW9uZXMvMTAxMTQ4OTUvY2lyY3VsYXJlcy1leHRlcm5hcy0yMDI0LyIsCiAgICAgICAgImh0dHBzOi8vd3d3LnN1cGVyZmluYW5jaWVyYS5nb3YuY28vcHVibGljYWNpb25lcy8xMDA5ODkyOS9jaXJjdWxhcmVzLWV4dGVybmFzLTIwMjMvIiwKICAgICAgICAiaHR0cHM6Ly93d3cuc3VwZXJmaW5hbmNpZXJhLmdvdi5jby9wdWJsaWNhY2lvbmVzLzEwMTE0ODMwL3Jlc29sdWNpb25lcy0yMDI0LyIsCiAgICAgICAgImh0dHBzOi8vd3d3LnN1cGVyZmluYW5jaWVyYS5nb3YuY28vcHVibGljYWNpb25lcy8xMDA5ODkzMy9yZXNvbHVjaW9uZXMtMjAyMy8iLAogICAgICAgICJodHRwczovL3d3dy5zdXBlcmZpbmFuY2llcmEuZ292LmNvL3B1YmxpY2FjaW9uZXMvMTAxMTQ4NzIvY2FydGFzLWNpcmN1bGFyZXMtMjAyNC8iLAogICAgXQoKICAgIEJBU0VfVVJMID0gImh0dHBzOi8vd3d3LnN1cGVyZmluYW5jaWVyYS5nb3YuY28iCgogICAgIyDilIDilIAgRXhwcmVzaW9uZXMgcGFyYSBpbmZlcmlyIHRpcG8gZGUgbm9ybWEgZGVsIG5vbWJyZSDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgIFRJUE9fUEFUVEVSTlMgPSBbCiAgICAgICAgKHJlLmNvbXBpbGUociJjaXJjdWxhciBleHRlcm5hIiwgcmUuSSksICJDaXJjdWxhcmVzIiksCiAgICAgICAgKHJlLmNvbXBpbGUociJjYXJ0YSBjaXJjdWxhciIsIHJlLkkpLCAgICJDaXJjdWxhcmVzIiksCiAgICAgICAgKHJlLmNvbXBpbGUociJyZXNvbHVjaVtvw7NdbiIsIHJlLkkpLCAgICAiUmVzb2x1Y2nDs24iKSwKICAgICAgICAocmUuY29tcGlsZShyImNvbmNlcHRvIiwgcmUuSSksICAgICAgICAgICJSZXNvbHVjacOzbiIpLAogICAgICAgIChyZS5jb21waWxlKHIiZGVjcmV0byIsIHJlLkkpLCAgICAgICAgICAgIkRlY3JldG8iKSwKICAgIF0KCiAgICBkZWYgZmV0Y2hfaW5kZXgoc2VsZikgLT4gbGlzdFtkaWN0XToKICAgICAgICBpdGVtcywgc2VlbiA9IFtdLCBzZXQoKQogICAgICAgIGZvciBpZHhfdXJsIGluIHNlbGYuSU5ERVhfVVJMUzoKICAgICAgICAgICAgcmVzcCA9IHNlbGYuX2dldChpZHhfdXJsKQogICAgICAgICAgICBpZiBub3QgcmVzcDoKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIHNvdXAgPSBzZWxmLl9zb3VwKHJlc3AudGV4dCkKICAgICAgICAgICAgZm9yIGEgaW4gc291cC5maW5kX2FsbCgiYSIsIGhyZWY9VHJ1ZSk6CiAgICAgICAgICAgICAgICBocmVmID0gYVsiaHJlZiJdCiAgICAgICAgICAgICAgICBpZiBub3QgaHJlZi5zdGFydHN3aXRoKCJodHRwIik6CiAgICAgICAgICAgICAgICAgICAgaHJlZiA9IHNlbGYuQkFTRV9VUkwgKyBocmVmCiAgICAgICAgICAgICAgICAjIFNvbG8gVVJMcyBkZSBwdWJsaWNhY2lvbmVzIGluZGl2aWR1YWxlcyAobm8gZWwgbWlzbW8gbGlzdGFkbykKICAgICAgICAgICAgICAgIGlmICgKICAgICAgICAgICAgICAgICAgICAiL3B1YmxpY2FjaW9uZXMvIiBpbiBocmVmCiAgICAgICAgICAgICAgICAgICAgYW5kIGhyZWYucnN0cmlwKCIvIikgIT0gaWR4X3VybC5yc3RyaXAoIi8iKQogICAgICAgICAgICAgICAgICAgIGFuZCBocmVmIG5vdCBpbiBzZWVuCiAgICAgICAgICAgICAgICApOgogICAgICAgICAgICAgICAgICAgIHNlZW4uYWRkKGhyZWYpCiAgICAgICAgICAgICAgICAgICAgaXRlbXMuYXBwZW5kKHsKICAgICAgICAgICAgICAgICAgICAgICAgInVybCI6ICAgICAgICBocmVmLAogICAgICAgICAgICAgICAgICAgICAgICAidGl0bGVfaGludCI6IGEuZ2V0X3RleHQoc3RyaXA9VHJ1ZSlbOjEyMF0sCiAgICAgICAgICAgICAgICAgICAgfSkKICAgICAgICAgICAgc2VsZi5sb2cuZGVidWcoZiJTRkMgw61uZGljZSB7aWR4X3VybH06IHtsZW4oc2Vlbil9IGFjdW11bGFkb3MiKQogICAgICAgIHJldHVybiBpdGVtcwoKICAgIGRlZiBfaW5mZXJfdGlwbyhzZWxmLCBub21icmU6IHN0cikgLT4gc3RyOgogICAgICAgIGZvciBwYXR0ZXJuLCB0aXBvIGluIHNlbGYuVElQT19QQVRURVJOUzoKICAgICAgICAgICAgaWYgcGF0dGVybi5zZWFyY2gobm9tYnJlKToKICAgICAgICAgICAgICAgIHJldHVybiB0aXBvCiAgICAgICAgcmV0dXJuICJDaXJjdWxhcmVzIgoKICAgIGRlZiBwYXJzZShzZWxmLCBodG1sOiBzdHIsIHVybDogc3RyKSAtPiBPcHRpb25hbFtkaWN0XToKICAgICAgICBzb3VwID0gc2VsZi5fc291cChodG1sKQoKICAgICAgICAjIOKUgOKUgCBOb21icmUg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICAgICAgbm9tYnJlID0gIiIKICAgICAgICBmb3Igc2VsIGluIFsiaDEudGl0dWxvLXB1YmxpY2FjaW9uIiwgImgxIiwgInRpdGxlIl06CiAgICAgICAgICAgIHRhZyA9IHNvdXAuc2VsZWN0X29uZShzZWwpCiAgICAgICAgICAgIGlmIHRhZzoKICAgICAgICAgICAgICAgIG5vbWJyZSA9IHRhZy5nZXRfdGV4dChzdHJpcD1UcnVlKQogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICBpZiBub3Qgbm9tYnJlOgogICAgICAgICAgICByZXR1cm4gTm9uZQoKICAgICAgICAjIOKUgOKUgCBGZWNoYSDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgICAgICBmZWNoYSA9ICIiCiAgICAgICAgZm9yIHNlbCBpbiBbCiAgICAgICAgICAgICJtZXRhW3Byb3BlcnR5PSdhcnRpY2xlOnB1Ymxpc2hlZF90aW1lJ10iLAogICAgICAgICAgICAidGltZVtkYXRldGltZV0iLAogICAgICAgICAgICAic3Bhbi5mZWNoYS1wdWJsaWNhY2lvbiIsCiAgICAgICAgICAgICJkaXYuZmVjaGEiLAogICAgICAgICAgICAic3Bhbi5mZWNoYSIsCiAgICAgICAgXToKICAgICAgICAgICAgdGFnID0gc291cC5zZWxlY3Rfb25lKHNlbCkKICAgICAgICAgICAgaWYgdGFnOgogICAgICAgICAgICAgICAgcmF3ID0gdGFnLmdldCgiY29udGVudCIpIG9yIHRhZy5nZXQoImRhdGV0aW1lIikgb3IgdGFnLmdldF90ZXh0KCkKICAgICAgICAgICAgICAgIGlmIHJhdzoKICAgICAgICAgICAgICAgICAgICBmZWNoYSA9IHNlbGYuX25vcm1fZGF0ZShyYXcuc3RyaXAoKVs6MzBdKQogICAgICAgICAgICAgICAgICAgIGJyZWFrCgogICAgICAgICMg4pSA4pSAIFRleHRvIGNvbXBsZXRvIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgICAgIHRleHRvID0gIiIKICAgICAgICBmb3Igc2VsIGluIFsKICAgICAgICAgICAgImRpdi5wdWJsaWNhY2lvbi1jb250ZW5pZG8iLAogICAgICAgICAgICAiZGl2LmNvbnRlbnQtYm9keSIsCiAgICAgICAgICAgICJkaXYuZmllbGQtaXRlbSIsCiAgICAgICAgICAgICJtYWluIGFydGljbGUiLAogICAgICAgICAgICAiYXJ0aWNsZSIsCiAgICAgICAgICAgICJtYWluIiwKICAgICAgICBdOgogICAgICAgICAgICB0YWcgPSBzb3VwLnNlbGVjdF9vbmUoc2VsKQogICAgICAgICAgICBpZiB0YWcgYW5kIGxlbih0YWcuZ2V0X3RleHQoc3RyaXA9VHJ1ZSkpID4gODA6CiAgICAgICAgICAgICAgICB0ZXh0byA9IHRhZy5nZXRfdGV4dChzZXBhcmF0b3I9IiAiLCBzdHJpcD1UcnVlKQogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICBpZiBub3QgdGV4dG86CiAgICAgICAgICAgIHRleHRvID0gc291cC5nZXRfdGV4dChzZXBhcmF0b3I9IiAiLCBzdHJpcD1UcnVlKQoKICAgICAgICByZXR1cm4gc2VsZi5fYnVpbGRfcmVjb3JkKAogICAgICAgICAgICBub21icmU9bm9tYnJlLAogICAgICAgICAgICB0aXBvX25vcm1hPXNlbGYuX2luZmVyX3RpcG8obm9tYnJlKSwKICAgICAgICAgICAgYXV0b3JpZGFkPSJTdXBlcmludGVuZGVuY2lhIEZpbmFuY2llcmEgZGUgQ29sb21iaWEiLAogICAgICAgICAgICBmZWNoYV9leHBlZGljaW9uPWZlY2hhLAogICAgICAgICAgICB1cmw9dXJsLAogICAgICAgICAgICB0ZXh0b19jb21wbGV0bz10ZXh0bywKICAgICAgICApCg==' 
#_content = base64.b64decode(_b64).decode('utf-8') 
#(PROJECT_ROOT / 'src/scraper/sfc_scraper.py').write_text(_content, encoding='utf-8') 
#print('\u2713 src/scraper/sfc_scraper.py') 


In [77]:
import sys, importlib
for ruta in (str(PROJECT_ROOT / "src"), str(PROJECT_ROOT / "src" / "scraper")):
    if ruta not in sys.path:
        sys.path.insert(0, ruta)

import sfc_scraper as _m; importlib.reload(_m)
from sfc_scraper import SFCScraper

print("SFC Scraper — test con max_docs=3")
print("─" * 50)

sfc = SFCScraper()

# Un solo llamado a run() — ya incluye fetch_index internamente
resultados = sfc.run(max_docs=3)

print(f"\n✓ Normas procesadas: {len(resultados)}")
for r in resultados:
    print(f"  ─ {r['nombre'][:70]}")
    print(f"    Tipo: {r['tipo_norma']} | Fecha: {r['fecha_expedicion']}")
    print(f"    URL : {r['url'][:80]}")
    print(f"    Texto: {len(r['texto_completo'])} chars")


10:02:54 | INFO     | sfcscraper | [SFC] Iniciando — modo test: max_docs=3
10:02:54 | INFO     | sfcscraper | SFC: leyendo tabla → https://www.superfinanciera.gov.co/publicaciones/10115974/circulares-externas-2026/


SFC Scraper — test con max_docs=3
──────────────────────────────────────────────────


10:03:11 | INFO     | sfcscraper | SFC: 6 normas acumuladas
10:03:11 | INFO     | sfcscraper | SFC: leyendo tabla → https://www.superfinanciera.gov.co/publicaciones/10115976/resoluciones-2026/
10:03:27 | INFO     | sfcscraper | SFC: 56 normas acumuladas
10:03:27 | INFO     | sfcscraper | SFC: leyendo tabla → https://www.superfinanciera.gov.co/publicaciones/10115975/cartas-circulares-2026/
10:03:43 | INFO     | sfcscraper | SFC: 93 normas acumuladas
10:03:43 | INFO     | sfcscraper | [SFC] 93 normas en la tabla
10:03:48 | INFO     | sfcscraper | [SFC] Finalizado — 3 OK, 0 errores



✓ Normas procesadas: 3
  ─ Circulares 006 de 2026 — Amplía el plazo de la entrada en vigencia de 
    Tipo: Circulares | Fecha: 11/05/2026
    URL : https://www.superfinanciera.gov.co/loader.php?lServicio=Tools2&lTipo=descargas&l
    Texto: 104 chars
  ─ Circulares 005 de 2026 — Imparte instrucciones para mitigar los efecto
    Tipo: Circulares | Fecha: 15/04/2026
    URL : https://www.superfinanciera.gov.co/loader.php?lServicio=Tools2&lTipo=descargas&l
    Texto: 288 chars
  ─ Circulares 004 de 2026 — Depura la Circular Básica Contable y Financie
    Tipo: Circulares | Fecha: 14/04/2026
    URL : https://www.superfinanciera.gov.co/loader.php?lServicio=Tools2&lTipo=descargas&l
    Texto: 113 chars


## BLOQUE 3 — `banrep_scraper.py` (Banco de la República)

**Fuente:** https://www.banrep.gov.co

**Qué extrae:**
- Circulares de la Junta Directiva
- Resoluciones
- Decretos relacionados

**Tecnología:** requests + BeautifulSoup (HTML estático, sin JavaScript)

In [78]:
import importlib
import banrep_scraper as _m; importlib.reload(_m)
from banrep_scraper import BanRepScraper

print("BanRep Scraper — modo producción (todas las normas 2026)")
print("─" * 50)
print("⚠ Se abrirá una ventana de Edge — puede tardar varios minutos")

br = BanRepScraper()
resultados = br.run(max_docs=0)

print(f"\n✓ Total normas 2026: {len(resultados)}")

# Resumen por tipo
tipos = {}
for r in resultados:
    t = r['tipo_norma']
    tipos[t] = tipos.get(t, 0) + 1

print("\nPor tipo:")
for t, c in sorted(tipos.items(), key=lambda x: -x[1]):
    print(f"  {t:35s}: {c}")

10:03:48 | INFO     | banrepscraper | [BANREP] Iniciando — producción
10:03:48 | INFO     | banrepscraper | BANREP: abriendo Edge visible (Radware requiere navegador real)


BanRep Scraper — modo producción (todas las normas 2026)
──────────────────────────────────────────────────
⚠ Se abrirá una ventana de Edge — puede tardar varios minutos


10:03:49 | INFO     | banrepscraper | BANREP: página 0 → https://www.banrep.gov.co/es/normatividad?page=0
10:03:57 | INFO     | banrepscraper | BANREP: 6 normas acumuladas
10:03:59 | INFO     | banrepscraper | BANREP: página 1 → https://www.banrep.gov.co/es/normatividad?page=1
10:04:04 | INFO     | banrepscraper | BANREP: 12 normas acumuladas
10:04:06 | INFO     | banrepscraper | BANREP: página 2 → https://www.banrep.gov.co/es/normatividad?page=2
10:04:12 | INFO     | banrepscraper | BANREP: 18 normas acumuladas
10:04:14 | INFO     | banrepscraper | BANREP: página 3 → https://www.banrep.gov.co/es/normatividad?page=3
10:04:19 | INFO     | banrepscraper | BANREP: 24 normas acumuladas
10:04:21 | INFO     | banrepscraper | BANREP: página 4 → https://www.banrep.gov.co/es/normatividad?page=4
10:04:27 | INFO     | banrepscraper | BANREP: 30 normas acumuladas
10:04:29 | INFO     | banrepscraper | BANREP: página 5 → https://www.banrep.gov.co/es/normatividad?page=5
10:04:35 | INFO     | banrepsc


✓ Total normas 2026: 58

Por tipo:
  Circular Externa                   : 42
  Boletín Junta Directiva            : 16


## BLOQUE 4 — `urf_scraper.py` (Unidad de Regulación Financiera)

**Fuente:** https://www.urf.gov.co/

**Qué extrae:**
- Circulares
- Resoluciones
- Regulaciones financieras
- Normatividad del sector financiero colombiano

**Tecnología:** por confirmar tras diagnóstico (requests vs Selenium)

In [79]:
# TEST URF SCRAPER
import importlib
import sys

for ruta in (str(PROJECT_ROOT / "src"), str(PROJECT_ROOT / "src" / "scraper")):
    if ruta not in sys.path:
        sys.path.insert(0, ruta)

import urf_scraper as _m; importlib.reload(_m)
from urf_scraper import URFScraper

print("URF Scraper — test con max_docs=0 (todas)")
print("─" * 50)

urf = URFScraper()
resultados = urf.run(max_docs=0)

print(f"\n✓ Decretos extraídos: {len(resultados)}")
for r in resultados[:5]:
    print(f"  ─ {r['nombre']}")
    print(f"    Fecha: {r['fecha_expedicion']} | URL: {r['url'][:80]}")

10:05:10 | INFO     | urfscraper | [URF] Iniciando — producción


URF Scraper — test con max_docs=0 (todas)
──────────────────────────────────────────────────


10:05:14 | INFO     | urfscraper | [URF] 7 decretos extraídos
10:05:14 | INFO     | urfscraper | [URF] Finalizado — 7 normas OK



✓ Decretos extraídos: 7
  ─ Decreto 0508 de 2026
    Fecha: 19/05/2026 | URL: https://www.urf.gov.co/documents/283253/3083471/DECRETO+No.+0508+DEL+19+DE+MAYO+
  ─ Decreto 0510 de 2026
    Fecha: 19/05/2026 | URL: https://www.urf.gov.co/documents/283253/3083471/DECRETO+510+DEL+19+DE+MAYO+DE+20
  ─ Decreto 0451 de 2026
    Fecha: 27/04/2026 | URL: https://www.urf.gov.co/documents/283253/3083471/DECRETO+No.+0451+DEL+27+DE+ABRIL
  ─ Decreto 0368 de 2026
    Fecha: 07/04/2026 | URL: https://www.urf.gov.co/documents/283253/3083471/DECRETO+No.+0368+DEL+07+DE+ABRIL
  ─ Decreto 0369 de 2026
    Fecha: 07/04/2026 | URL: https://www.urf.gov.co/documents/283253/3083471/DECRETO+No.+0369+DEL+07+DE+ABRIL


## BLOQUE 5 — `minhac_scraper.py` (Ministerio de Hacienda y Crédito Público)

**Fuente:** https://www.minhacienda.gov.co

**Qué extrae:**
- Decretos 2026
- Resoluciones y Circulares 2026

**URLs directas:**
- https://www.minhacienda.gov.co/decretos-2026
- https://www.minhacienda.gov.co/resoluciones-2026

**Tecnología:** por confirmar tras diagnóstico (requests vs Selenium)

In [90]:
# TEST MINHAC SCRAPER
import importlib, sys

for ruta in (str(PROJECT_ROOT / "src"), str(PROJECT_ROOT / "src" / "scraper")):
    if ruta not in sys.path:
        sys.path.insert(0, ruta)

import minhac_scraper as _m; importlib.reload(_m)
from minhac_scraper import MinhacScraper

print("MinHac Scraper — test con max_docs=0 (todas)")
print("─" * 50)

mh = MinhacScraper()
resultados = mh.run(max_docs=0)

print(f"\n✓ Normas extraídas: {len(resultados)}")
for r in resultados[:5]:
    print(f"  ─ {r['nombre'][:70]}")
    print(f"    Tipo: {r['tipo_norma']} | Fecha: {r['fecha_expedicion']}")

10:39:13 | INFO     | minhacscraper | [MINHAC] Iniciando — producción
10:39:13 | INFO     | minhacscraper | MINHAC: scraping Decreto → https://www.minhacienda.gov.co/decretos-2026


MinHac Scraper — test con max_docs=0 (todas)
──────────────────────────────────────────────────


10:39:16 | INFO     | minhacscraper | MINHAC: 20 Decretos extraídos
10:39:16 | INFO     | minhacscraper | MINHAC: scraping Resolución → https://www.minhacienda.gov.co/resoluciones-2026
10:39:18 | WARNING  | minhacscraper | MINHAC: tabla no encontrada para Resolución
10:39:18 | INFO     | minhacscraper | MINHAC: 0 Resolucións extraídos
10:39:18 | INFO     | minhacscraper | [MINHAC] Finalizado — 20 normas OK



✓ Normas extraídas: 20
  ─ DECRETO No. 0549 DEL 01 DE JUNIO DE 2026
    Tipo: Decreto | Fecha: 01/06/2026
  ─ DECRETO No. 0536 DEL 25 DE MAYO DE 2026
    Tipo: Decreto | Fecha: 25/05/2026
  ─ DECRETO No. 0510 DEL 19 DE MAYO DE 2026
    Tipo: Decreto | Fecha: 19/05/2026
  ─ DECRETO No. 0509 DEL 19 DE MAYO DE 2026
    Tipo: Decreto | Fecha: 19/05/2026
  ─ DECRETO No. 0508 DEL 19 DE MAYO DE 2026
    Tipo: Decreto | Fecha: 19/05/2026


In [91]:
import importlib
import scraper_manager as _sm; importlib.reload(_sm)
from scraper_manager import SCRAPER_REGISTRY, run_all

print("Scrapers registrados:")
for codigo, clase in SCRAPER_REGISTRY.items():
    print(f"  {codigo:10s} → {clase.__name__}")

Scrapers registrados:
  SFC        → SFCScraper
  BANREP     → BanRepScraper
  URF        → URFScraper
  MINHAC     → MinhacScraper


In [80]:
# Verificar que scraper_manager.py existe
ruta = PROJECT_ROOT / 'src/scraper/scraper_manager.py'
print(f"✓ src/scraper/scraper_manager.py" if ruta.exists() else "✗ FALTA src/scraper/scraper_manager.py")

✓ src/scraper/scraper_manager.py


In [81]:
# ── Test scraper_manager: verificar que todos los scrapers registran ─ 
import importlib, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
for ruta in [
    str(PROJECT_ROOT),
    str(PROJECT_ROOT / "src"),
    str(PROJECT_ROOT / "src" / "scraper"),
    str(PROJECT_ROOT / "src" / "storage"),
]:
    if ruta not in sys.path:
        sys.path.insert(0, ruta)

import scraper_manager as _sm; importlib.reload(_sm)
from scraper_manager import SCRAPER_REGISTRY, run_all

print("Scrapers registrados en SCRAPER_REGISTRY:")
for codigo, clase in SCRAPER_REGISTRY.items():
    print(f"  {codigo:10s} → {clase.__name__}")

print(f"\nFuentes activas en config: {list(config.fuentes_activas().keys())}")
print("\n✓ scraper_manager.py listo")
print("  Para ejecutar todos: run_all(max_docs_per_source=3) [modo test]")
print("  Para producción   : run_all()                       [sin límite]")

Scrapers registrados en SCRAPER_REGISTRY:
  SFC        → SFCScraper
  BANREP     → BanRepScraper
  URF        → URFScraper

Fuentes activas en config: ['SFC', 'BANREP', 'MINHAC', 'DIAN', 'URF']

✓ scraper_manager.py listo
  Para ejecutar todos: run_all(max_docs_per_source=3) [modo test]
  Para producción   : run_all()                       [sin límite]


In [82]:
# Verificar que raw_storage.py existe
ruta = PROJECT_ROOT / 'src/storage/raw_storage.py'
print(f"✓ src/storage/raw_storage.py" if ruta.exists() else "✗ FALTA src/storage/raw_storage.py")

✓ src/storage/raw_storage.py


In [83]:
# Verificar que change_detector.py existe
ruta = PROJECT_ROOT / 'src/storage/change_detector.py'
print(f"✓ src/storage/change_detector.py" if ruta.exists() else "✗ FALTA src/storage/change_detector.py")

✓ src/storage/change_detector.py


In [84]:
# Verificar que deduplicator.py existe
ruta = PROJECT_ROOT / 'src/storage/deduplicator.py'
print(f"✓ src/storage/deduplicator.py" if ruta.exists() else "✗ FALTA src/storage/deduplicator.py")

✓ src/storage/deduplicator.py


In [ ]:
# ════════════════════════════════════════════════════════
# EJECUCIÓN COMPLETA
# MAX_DOCS = 5  → modo test (rápido)
# MAX_DOCS = 0  → modo producción (todas las normas)
# ════════════════════════════════════════════════════════

MAX_DOCS = 0

import importlib, sys
for ruta in (str(PROJECT_ROOT / 'src'), str(PROJECT_ROOT / 'src' / 'scraper'),
             str(PROJECT_ROOT / 'src' / 'storage')):
    if ruta not in sys.path:
        sys.path.insert(0, ruta)

import scraper_manager as sm_mod; importlib.reload(sm_mod)
import raw_storage     as rs_mod; importlib.reload(rs_mod)
import change_detector as cd_mod; importlib.reload(cd_mod)
import deduplicator    as dd_mod; importlib.reload(dd_mod)

from scraper_manager import run_all
from raw_storage     import save, load_latest
from change_detector import detect
from deduplicator    import normalize, deduplicate
import pandas as pd

print('=' * 58)
print('PIPELINE DE SCRAPING — Paso 2')
print('=' * 58)
print(f"Modo: {'TEST (max ' + str(MAX_DOCS) + ' por fuente)' if MAX_DOCS else 'PRODUCCION (sin limite)'}")

# PASO 1: Scraping
print('\nPASO 1: Ejecutando scrapers...')
normas_raw = run_all(max_docs_per_source=MAX_DOCS)
print(f'  Total extraídas: {len(normas_raw)} normas')

if not normas_raw:
    print('\n⚠  No se extrajeron normas. Revisa el test de conectividad.')
else:
    # PASO 2: Normalizar y deduplicar
    print('\nPASO 2: Normalizando y deduplicando...')
    df = pd.DataFrame(normas_raw)
    df = normalize(df)
    df = deduplicate(df)
    print(f'  Después de dedup: {len(df)} normas')

    # PASO 3: Detectar cambios
    print('\nPASO 3: Detectando cambios...')
    df_prev = load_latest()
    df = detect(df, df_prev)
    estados = df['estado_cambio'].value_counts().to_dict()
    for e, c in estados.items():
        print(f'  {e:15s}: {c}')

    # PASO 4: Guardar Excel
    print('\nPASO 4: Guardando Excel crudo...')
    ruta = save(df.to_dict(orient='records'))
    if ruta:
        print(f'  ✓ {ruta}')

    print(f'\nTOTAL: {len(df)} normas')
    print('=' * 58)


10:38:29 | INFO     | scraper_manager | Scrapers a ejecutar: ['SFC', 'BANREP', 'MINHAC', 'DIAN', 'URF']
10:38:29 | INFO     | scraper_manager | 
──────────────────────────────────────────────────
10:38:29 | INFO     | scraper_manager | Iniciando: Superintendencia Financiera de Colombia
10:38:29 | INFO     | sfcscraper | [SFC] Iniciando
10:38:29 | INFO     | sfcscraper | SFC: leyendo tabla → https://www.superfinanciera.gov.co/publicaciones/10115974/circulares-externas-2026/


PIPELINE DE SCRAPING — Paso 2
Modo: PRODUCCION (sin limite)

PASO 1: Ejecutando scrapers...


KeyboardInterrupt: 